# Homework: The EM Algorithm

## Problem 1: EM for a Two-Component Exponential Mixture

Suppose you observe $n$ data points $y_1, \ldots, y_n$ from a two-component exponential mixture:

$$f(y_i | \boldsymbol{\theta}) = \pi \, \lambda_1 e^{-\lambda_1 y_i} + (1 - \pi) \, \lambda_2 e^{-\lambda_2 y_i}, \quad y_i > 0$$

where $\boldsymbol{\theta} = (\pi, \lambda_1, \lambda_2)$.

**(a)** Write down the complete-data log-likelihood, treating the component membership $z_i \in \{1, 2\}$ as the latent variable.

**(b)** Derive the E-step: compute the responsibility $\gamma_i^{(t)} = P(z_i = 1 | y_i, \boldsymbol{\theta}^{(t)})$.

**(c)** Derive the M-step: write the update formulas for $\pi^{(t+1)}$, $\lambda_1^{(t+1)}$, and $\lambda_2^{(t+1)}$.

**(d)** Implement the EM algorithm for this model. Your function should have the following signature and return a dictionary containing the estimated parameters and the log-likelihood history.

Let indicator $x_i = \mathbf{1}(z_i = 1)$, so $x_i \in \{0,1\}$ and $1-x_i = \mathbf{1}(z_i=2)$.

**(a) Complete-data log-likelihood**

The complete-data likelihood is
$$
L_c(\theta)=\prod_{i=1}^n \Big[\pi\,\lambda_1 e^{-\lambda_1 y_i}\Big]^{x_i}
\Big[(1-\pi)\,\lambda_2 e^{-\lambda_2 y_i}\Big]^{1-x_i}.
$$
So,
$$
\ell_c(\theta)=\sum_{i=1}^n \left\{x_i\big(\log\pi+\log\lambda_1-\lambda_1 y_i\big)
+(1-x_i)\big(\log(1-\pi)+\log\lambda_2-\lambda_2 y_i\big)
\right\}.
$$

**(b) E-step**

At iteration $t$, compute
$$
\gamma_i^{(t)}=P(z_i=1\mid y_i, \theta^{(t)})
=\frac{\pi^{(t)}\lambda_1^{(t)}e^{-\lambda_1^{(t)}y_i}}
{\pi^{(t)}\lambda_1^{(t)}e^{-\lambda_1^{(t)}y_i}+(1-\pi^{(t)})\lambda_2^{(t)}e^{-\lambda_2^{(t)}y_i}}.
$$
Then $P(z_i=2\mid y_i, \theta^{(t)})=1-\gamma_i^{(t)}$.

**(c) M-step**

Maximizing the expected complete log-likelihood gives:
$$
\pi^{(t+1)}=\frac{1}{n}\sum_{i=1}^n \gamma_i^{(t)},
$$
$$
\lambda_1^{(t+1)}=\frac{\sum_{i=1}^n \gamma_i^{(t)}}{\sum_{i=1}^n \gamma_i^{(t)} y_i},
\qquad
\lambda_2^{(t+1)}=\frac{\sum_{i=1}^n (1-\gamma_i^{(t)})}{\sum_{i=1}^n (1-\gamma_i^{(t)}) y_i}.
$$



In [1]:
import numpy as np
from scipy import stats


def em_exponential_mixture(y, pi_init, lam1_init, lam2_init,
                           max_iter=200, tol=1e-8):
    """EM algorithm for a 2-component exponential mixture.

    Parameters
    ----------
    y : np.ndarray
        Observed data of shape (n,), all positive.
    pi_init : float
        Initial mixing proportion for component 1.
    lam1_init, lam2_init : float
        Initial rate parameters.
    max_iter : int
        Maximum number of iterations.
    tol : float
        Convergence tolerance on log-likelihood change.

    Returns
    -------
    dict with keys: pi, lam1, lam2, loglik_history
    """
    y = np.asarray(y, dtype=float)
    if y.ndim != 1:
        raise ValueError("y must be a 1D array")
    if np.any(y <= 0):
        raise ValueError("All y values must be > 0")

    pi = float(pi_init)
    lam1 = float(lam1_init)
    lam2 = float(lam2_init)

    if not (0.0 < pi < 1.0):
        raise ValueError("pi_init must be in (0, 1)")
    if lam1 <= 0 or lam2 <= 0:
        raise ValueError("lam1_init and lam2_init must be > 0")

    eps = 1e-12
    loglik_history = []

    for _ in range(max_iter):
        # E-step: responsibilities for component 1
        f1 = stats.expon.pdf(y, scale=1.0 / lam1)
        f2 = stats.expon.pdf(y, scale=1.0 / lam2)
        denom = pi * f1 + (1.0 - pi) * f2 + eps
        gamma = (pi * f1) / denom

        # M-step
        w1 = np.sum(gamma)
        w2 = np.sum(1.0 - gamma)

        pi = np.clip(w1 / y.size, eps, 1.0 - eps)
        lam1 = max(w1 / np.sum(gamma * y), eps)
        lam2 = max(w2 / np.sum((1.0 - gamma) * y), eps)

        # Observed-data log-likelihood under updated parameters
        mix = pi * stats.expon.pdf(y, scale=1.0 / lam1) + (1.0 - pi) * stats.expon.pdf(y, scale=1.0 / lam2)
        loglik = np.sum(np.log(mix + eps))
        loglik_history.append(loglik)

        if len(loglik_history) > 1 and abs(loglik_history[-1] - loglik_history[-2]) < tol:
            break

    return {
        "pi": pi,
        "lam1": lam1,
        "lam2": lam2,
        "loglik_history": loglik_history,
    }



/Users/wenbinwu/miniforge3/lib/python3.9/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.26.0 is required for this version of SciPy (detected version 1.26.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Test your implementation on the following simulated data and report the estimated parameters:

In [2]:
np.random.seed(123)
n = 500
pi_true = 0.35
lam1_true, lam2_true = 0.5, 3.0
z = np.random.binomial(1, 1 - pi_true, n)
y = np.where(z == 0,
             np.random.exponential(1 / lam1_true, n),
             np.random.exponential(1 / lam2_true, n))

result = em_exponential_mixture(y, pi_init=0.5, lam1_init=0.3, lam2_init=2.0)

In [3]:
print(f"estimated parameters: pi = {result['pi']:.4f}, lam1 = {result['lam1']:.4f}, lam2 = {result['lam2']:.4f}")

estimated parameters: pi = 0.2992, lam1 = 0.4354, lam2 = 2.4793


## Problem 2: Diagnosing EM Convergence

A colleague provides the following EM implementation for the two-component Gaussian mixture model. Read the code carefully and answer the questions below.

In [4]:
import numpy as np
from scipy import stats

def em_gmm_v2(y, pi_init, mu1_init, sigma1_init, mu2_init, sigma2_init,
              max_iter=200, tol=1e-8):
    n = len(y)
    pi = pi_init
    mu1, mu2 = mu1_init, mu2_init
    s1, s2 = sigma1_init, sigma2_init
    loglik_history = []

    for iteration in range(max_iter):
        d1 = pi * stats.norm.pdf(y, mu1, s1)
        d2 = (1 - pi) * stats.norm.pdf(y, mu2, s2)
        gamma = d1 / (d1 + d2)

        ll = np.sum(np.log(d1 + d2))
        loglik_history.append(ll)

        if iteration > 0 and loglik_history[-1] - loglik_history[-2] < tol:
            break

        n1 = np.sum(gamma)
        n2 = n - n1

        pi = n1 / n
        mu1 = np.sum(gamma * y) / n1
        mu2 = np.sum((1 - gamma) * y) / n2
        s1 = np.sqrt(np.sum(gamma * (y - mu1) ** 2) / n1)
        s2 = np.sqrt(np.sum((1 - gamma) * (y - mu2) ** 2) / n2)

    return {
        "pi": pi, "mu1": mu1, "sigma1": s1,
        "mu2": mu2, "sigma2": s2,
        "loglik_history": loglik_history,
    }

**(a)** There is a subtle bug in the convergence check. Identify it and explain what incorrect behavior it could cause.

**(b)** Suppose this function is called with `sigma1_init=0.001` on a dataset where one observation happens to be very close to `mu1_init`. Explain what numerical issue could arise during the E-step and how you would fix it.

**(c)** Does this implementation handle the label-switching symmetry of the mixture model? That is, if you swap the roles of component 1 and component 2 in the initialization, do you get an equivalent solution? Explain why or why not.

(a) The check

```
if iteration > 0 and loglik_history[-1] - loglik_history[-2] < tol:
    break
```

will allow any decrease in the log-likelihood to stop the iteration (could be well before convergence). Change it to also require the improvement to be nonnegative:

```
delta = loglik_history[-1] - loglik_history[-2]
if iteration > 0 and 0 <= delta < tol:
    break
```

(b) The first distribution could be extremely peaked. `d1` or `d2` could underflow to 0, giving `gamma=NaN`. Responsiblities can collapase to 0 or 1 and cause unstable variance update.

Fix: Compute `d1` and `d2` in the log space.

(c) Yes. EM algorithm will follow a mirrored path and return the same fit. 


## Problem 3: EM for Missing Data

Consider a dataset of $n = 200$ paired measurements $(X_i, Y_i)$ from a bivariate normal distribution, where 25% of $Y$ values are missing at random.

**(a)** Write a function that implements the EM algorithm for estimating the bivariate normal parameters $(\mu_X, \mu_Y, \sigma_X^2, \sigma_Y^2, \rho)$ when $Y$ has missing values. Use the following signature:

In [5]:
def em_bivariate(X, Y, observed, max_iter=200, tol=1e-8):
    """EM for bivariate normal with missing Y values.

    Parameters
    ----------
    X : np.ndarray
        Fully observed variable, shape (n,).
    Y : np.ndarray
        Partially observed variable, shape (n,). NaN for missing.
    observed : np.ndarray
        Boolean array, True where Y is observed.
    max_iter : int
        Maximum iterations.
    tol : float
        Convergence tolerance on max parameter change.

    Returns
    -------
    dict with keys: mu_x, mu_y, var_x, var_y, rho, n_iter
    """
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    observed = np.asarray(observed, dtype=bool)

    if X.ndim != 1 or Y.ndim != 1 or observed.ndim != 1:
        raise ValueError("X, Y, and observed must be 1D arrays")
    if not (len(X) == len(Y) == len(observed)):
        raise ValueError("X, Y, and observed must have the same length")
    if np.any(observed & np.isnan(Y)):
        raise ValueError("observed=True entries cannot be NaN in Y")
    if np.sum(observed) < 2:
        raise ValueError("Need at least two observed Y values")

    n = len(X)
    miss = ~observed
    eps = 1e-12

    # Initialize from complete cases
    X_obs = X[observed]
    Y_obs = Y[observed]

    mu_x = np.mean(X)
    mu_y = np.mean(Y_obs)

    var_x = np.var(X, ddof=0)
    var_y = np.var(Y_obs, ddof=0)
    cov_xy = np.mean((X_obs - np.mean(X_obs)) * (Y_obs - np.mean(Y_obs)))

    var_x = max(var_x, eps)
    var_y = max(var_y, eps)

    for it in range(1, max_iter + 1):
        old = np.array([mu_x, mu_y, var_x, var_y, cov_xy], dtype=float)

        # E-step: conditional moments for missing Y | X
        beta = cov_xy / max(var_x, eps)
        cond_mean = mu_y + beta * (X[miss] - mu_x)
        cond_var = var_y - (cov_xy ** 2) / max(var_x, eps)
        cond_var = max(cond_var, eps)

        # Expected sufficient statistics
        Sx = np.sum(X)
        Sy = np.sum(Y_obs) + np.sum(cond_mean)
        Sxx = np.sum(X ** 2)
        Syy = np.sum(Y_obs ** 2) + np.sum(cond_var + cond_mean ** 2)
        Sxy = np.sum(X_obs * Y_obs) + np.sum(X[miss] * cond_mean)

        # M-step
        mu_x = Sx / n
        mu_y = Sy / n

        var_x = Sxx / n - mu_x ** 2
        var_y = Syy / n - mu_y ** 2
        cov_xy = Sxy / n - mu_x * mu_y

        var_x = max(var_x, eps)
        var_y = max(var_y, eps)

        new = np.array([mu_x, mu_y, var_x, var_y, cov_xy], dtype=float)
        if np.max(np.abs(new - old)) < tol:
            break

    rho = cov_xy / np.sqrt(var_x * var_y)
    rho = float(np.clip(rho, -1 + 1e-10, 1 - 1e-10))

    return {
        "mu_x": float(mu_x),
        "mu_y": float(mu_y),
        "var_x": float(var_x),
        "var_y": float(var_y),
        "rho": rho,
        "n_iter": int(it),
    }

**(b)** Test your implementation on the following simulated data. Compare the EM estimates with the complete-case estimates (using only observations where $Y$ is observed) and the full-data estimates (using the true $Y$ values before deletion).

In [6]:
np.random.seed(42)
n = 200
mu = np.array([3.0, 7.0])
rho_true = 0.6
sx, sy = 1.5, 2.0
Sigma = np.array([[sx**2, rho_true * sx * sy],
                   [rho_true * sx * sy, sy**2]])

data = np.random.multivariate_normal(mu, Sigma, n)
X, Y_full = data[:, 0], data[:, 1]

# Introduce 25% MAR missingness
prob_miss = 0.25 * np.ones(n)
missing = np.random.binomial(1, prob_miss, n).astype(bool)
Y = Y_full.copy()
Y[missing] = np.nan
observed = ~missing

In [7]:
# EM estimate
em = em_bivariate(X, Y, observed)

# Complete-case estimate (drop missing Y)
X_cc = X[observed]
Y_cc = Y[observed]

def mle_bivariate(x, y):
    mx, my = np.mean(x), np.mean(y)
    vx, vy = np.var(x, ddof=0), np.var(y, ddof=0)
    cov = np.mean((x - mx) * (y - my))
    rho = cov / np.sqrt(vx * vy)
    return {"mu_x": mx, "mu_y": my, "var_x": vx, "var_y": vy, "rho": rho}

cc = mle_bivariate(X_cc, Y_cc)
full = mle_bivariate(X, Y_full)
print("EM estimates:")
print(f"mu_x: {em['mu_x']:.4f}, mu_y: {em['mu_y']:.4f}, var_x: {em['var_x']:.4f}, var_y: {em['var_y']:.4f}, rho: {em['rho']:.4f}")
print("\nComplete-case estimates:")
print(f"mu_x: {cc['mu_x']:.4f}, mu_y: {cc['mu_y']:.4f}, var_x: {cc['var_x']:.4f}, var_y: {cc['var_y']:.4f}, rho: {cc['rho']:.4f}")
print("\nFull-data estimates:")
print(f"mu_x: {full['mu_x']:.4f}, mu_y: {full['mu_y']:.4f}, var_x: {full['var_x']:.4f}, var_y: {full['var_y']:.4f}, rho: {full['rho']:.4f}")

EM estimates:
mu_x: 2.9576, mu_y: 7.0211, var_x: 2.0088, var_y: 3.8267, rho: 0.5826

Complete-case estimates:
mu_x: 2.9163, mu_y: 6.9879, var_x: 2.0303, var_y: 3.8406, rho: 0.5847

Full-data estimates:
mu_x: 2.9576, mu_y: 7.0096, var_x: 2.0088, var_y: 3.7115, rho: 0.5892


**(c)** Which estimator do you expect to have smaller variance for $\rho$, the EM estimator or the complete-case estimator? Explain why.

EM estimator. Complete-case uses only observed pairs and discards the rest, so its effective sample size is smaller.

## Problem 4: Zero-Inflated Poisson Model Selection

The following code fits a standard Poisson model and a zero-inflated Poisson (ZIP) model to a dataset. Read the code and answer the questions.

In [8]:
import numpy as np
from scipy import stats

def fit_poisson(y):
    """Fit a standard Poisson model by MLE."""
    lam_hat = np.mean(y)
    ll = np.sum(stats.poisson.logpmf(y, lam_hat))
    return {"lam": lam_hat, "loglik": ll, "n_params": 1}

def fit_zip_em(y, max_iter=200, tol=1e-8):
    """Fit a zero-inflated Poisson model by EM."""
    n = len(y)
    pi = 0.3
    lam = np.mean(y[y > 0])
    is_zero = (y == 0)
    loglik_history = []

    for iteration in range(max_iter):
        gamma = np.zeros(n)
        gamma[is_zero] = pi / (pi + (1 - pi) * np.exp(-lam))

        ll = (np.sum(is_zero) * np.log(pi + (1 - pi) * np.exp(-lam))
              + np.sum(~is_zero * (np.log(1 - pi)
                                    + stats.poisson.logpmf(y, lam))))
        loglik_history.append(ll)

        if iteration > 0 and abs(loglik_history[-1] - loglik_history[-2]) < tol:
            break

        pi = np.mean(gamma)
        lam = np.sum((1 - gamma) * y) / np.sum(1 - gamma)

    return {"pi": pi, "lam": lam, "loglik": ll,
            "n_params": 2, "loglik_history": loglik_history}

**(a)** In the E-step, why is $\gamma_i = 0$ for all observations with $y_i > 0$? Explain using the structure of the ZIP model.

**(b)** Given that the Poisson model is nested within the ZIP model (setting $\pi = 0$ recovers the Poisson), a natural model comparison tool is the likelihood ratio test. However, the standard chi-squared approximation for the LRT is not valid here. Explain why, in terms of the parameter space and the null hypothesis.

(a) In ZIP, each observation comes from either a structural-zero component (always outputs 0) or a Poisson component. If $y_i > 0$, it can only come from the former, thus $\gamma_i=0$.

(b) The null hypothesis is $H_0: \pi = 0$. However, because $\pi \in [0,1]$, the null hypothesis is on the boundary of the parameter space. Hence the standard testing is invalid (requires null parameter value to be in the interior of whole parameter space).

**(c)** Using the AIC ($\text{AIC} = -2\ell + 2k$, where $k$ is the number of parameters), compare the Poisson and ZIP fits on the following data. Which model does AIC prefer?

In [9]:
np.random.seed(10)
n = 300
pi_true = 0.2
lam_true = 2.0
z = np.random.binomial(1, pi_true, n)
y = np.where(z == 1, 0, np.random.poisson(lam_true, n))

poisson_fit = fit_poisson(y)
zip_fit = fit_zip_em(y)

In [10]:
def aic(loglik, k):
    return -2 * loglik + 2 * k

# Compute AIC
aic_poisson = aic(poisson_fit["loglik"], poisson_fit["n_params"])
aic_zip = aic(zip_fit["loglik"], zip_fit["n_params"])

print("Poisson fit:", poisson_fit)
print("ZIP fit:", {k: v for k, v in zip_fit.items() if k != "loglik_history"})
print(f"AIC (Poisson): {aic_poisson:.4f}")
print(f"AIC (ZIP): {aic_zip:.4f}")

Poisson fit: {'lam': 1.52, 'loglik': -496.0909991283401, 'n_params': 1}
ZIP fit: {'pi': 0.182777331600003, 'lam': 1.859958195941797, 'loglik': -483.76273160559305, 'n_params': 2}
AIC (Poisson): 994.1820
AIC (ZIP): 971.5255


AIC prefers ZIP.

## Problem 5: EM with Asymmetric Components

Consider a two-component mixture where component 1 is $\text{Exponential}(\lambda)$ (supported on $[0, \infty)$) and component 2 is $N(\mu, \sigma^2)$ (supported on $(-\infty, \infty)$):

$$f(y_i | \boldsymbol{\theta}) = \pi \, \lambda e^{-\lambda y_i} \mathbb{1}(y_i > 0) + (1 - \pi) \, \phi(y_i | \mu, \sigma^2)$$

**(a)** Derive the E-step and M-step for this model. Pay careful attention to how the exponential component's support restriction affects the responsibility computation for observations with $y_i \leq 0$.

(a)

Let latent indicator $z_i \in \{0,1\}$ where $z_i=1$ means exponential component and $z_i=0$ means normal component.

E-step:
$$
\gamma_i = P(z_i=1\mid y_i,\theta)
$$

Because the exponential component has support only on $(0,\infty)$, we have $\gamma_i = 0$ if $y_i \le 0$, and if $y_i > 0$, then
$$
\gamma_i = \frac{\pi\,\lambda e^{-\lambda y_i}}
{\pi\,\lambda e^{-\lambda y_i} + (1-\pi)\,\phi(y_i\mid\mu,\sigma^2)}.
$$

M-step:
$$
\pi^{new} = \frac{1}{n}\sum_{i=1}^n \gamma_i,
$$
$$
\lambda^{new} = \frac{\sum_{i=1}^n \gamma_i}{\sum_{i=1}^n \gamma_i y_i},
$$
$$
\mu^{new} = \frac{\sum_{i=1}^n (1-\gamma_i) y_i}{\sum_{i=1}^n (1-\gamma_i)},
$$
$$
(\sigma^2)^{new} = \frac{\sum_{i=1}^n (1-\gamma_i)(y_i-\mu^{new})^2}{\sum_{i=1}^n (1-\gamma_i)}.
$$

Then $\sigma^{new} = \sqrt{(\sigma^2)^{new}}$.



**(b)** Implement the EM algorithm for this model:

In [11]:
def em_exp_normal_mixture(y, pi_init, lam_init, mu_init, sigma_init,
                          max_iter=200, tol=1e-8):
    """EM for exponential-normal mixture.

    Parameters
    ----------
    y : np.ndarray
        Observed data of shape (n,).
    pi_init : float
        Initial mixing proportion for the exponential component.
    lam_init : float
        Initial rate for the exponential component.
    mu_init : float
        Initial mean for the normal component.
    sigma_init : float
        Initial std dev for the normal component.
    max_iter : int
        Maximum iterations.
    tol : float
        Convergence tolerance on log-likelihood change.

    Returns
    -------
    dict with keys: pi, lam, mu, sigma, loglik_history
    """
    import numpy as np
    from scipy import stats

    y = np.asarray(y, dtype=float)
    if y.ndim != 1:
        raise ValueError("y must be a 1D array")

    pi = float(pi_init)
    lam = float(lam_init)
    mu = float(mu_init)
    sigma = float(sigma_init)

    if not (0.0 < pi < 1.0):
        raise ValueError("pi_init must be in (0, 1)")
    if lam <= 0:
        raise ValueError("lam_init must be > 0")
    if sigma <= 0:
        raise ValueError("sigma_init must be > 0")

    eps = 1e-12
    loglik_history = []

    for _ in range(max_iter):
        # E-step with support restriction for exponential component
        gamma = np.zeros_like(y, dtype=float)
        pos = y > 0

        exp_pdf = np.zeros_like(y, dtype=float)
        exp_pdf[pos] = stats.expon.pdf(y[pos], scale=1.0 / lam)
        norm_pdf = stats.norm.pdf(y, loc=mu, scale=sigma)

        denom = pi * exp_pdf + (1.0 - pi) * norm_pdf + eps
        gamma[pos] = (pi * exp_pdf[pos]) / denom[pos]
        # For y <= 0, gamma remains 0 by support of exponential

        # M-step
        w_exp = np.sum(gamma)
        w_norm = np.sum(1.0 - gamma)

        pi = np.clip(w_exp / y.size, eps, 1.0 - eps)
        lam = max(w_exp / max(np.sum(gamma * y), eps), eps)

        mu = np.sum((1.0 - gamma) * y) / max(w_norm, eps)
        var = np.sum((1.0 - gamma) * (y - mu) ** 2) / max(w_norm, eps)
        sigma = np.sqrt(max(var, eps))

        # Observed-data log-likelihood
        exp_part = np.zeros_like(y, dtype=float)
        exp_part[pos] = pi * lam * np.exp(-lam * y[pos])
        norm_part = (1.0 - pi) * stats.norm.pdf(y, loc=mu, scale=sigma)
        mix = exp_part + norm_part
        ll = np.sum(np.log(mix + eps))
        loglik_history.append(ll)

        if len(loglik_history) > 1 and abs(loglik_history[-1] - loglik_history[-2]) < tol:
            break

    return {
        "pi": float(pi),
        "lam": float(lam),
        "mu": float(mu),
        "sigma": float(sigma),
        "loglik_history": loglik_history,
    }

**(c)** Test on the following data and report your estimates:

In [12]:
np.random.seed(42)
n = 400
pi_true = 0.4
lam_true = 2.0
mu_true, sigma_true = -1.0, 1.5
z = np.random.binomial(1, pi_true, n)
y = np.where(z == 1,
             np.random.exponential(1 / lam_true, n),
             np.random.normal(mu_true, sigma_true, n))

result = em_exp_normal_mixture(y, pi_init=0.5, lam_init=1.0,
                                mu_init=0.0, sigma_init=2.0)

print(f"Estimated parameters:")
print(f"pi: {result['pi']:.4f}, lam: {result['lam']:.4f}, mu: {result['mu']:.4f}, sigma: {result['sigma']:.4f}")

Estimated parameters:
pi: 0.4224, lam: 1.8426, mu: -0.9123, sigma: 1.4838
